# Incident evidence

Trace a recorded monitoring minute into original metrics, recent logs, and saved-model evidence. The data and fitted artifacts remain outside Git.


In [ ]:
from pathlib import Path
from banking_observability.incident_evidence import read_evidence

root = Path.cwd()
evidence = read_evidence(root / "data/raw", as_of="2025-10-25 18:01:00")
print(evidence["as_of"], evidence["monitoring_rows"], evidence["recent_log_count"])

## A source-aligned minute

The median comes from the prior 60 minutes. It is a descriptive comparison, not an alert threshold.


In [ ]:
for metric in evidence["monitoring"]:
    print(metric["metric"], metric["observed"], metric["prior_60_minute_median"], metric["reference"]["csv_row"])

## Log records and references

Inspect the original wording and CSV row numbers. A matching catalog event type does not identify a downstream failure cause.


In [ ]:
for row in evidence["logs_most_recent_first"][:5]:
    print(row["csv_row"], row["timestamp"], row["log_level"], row["log_message"])

## Fitted-model and retrieval check

The CLI needs the existing anomaly, log, and knowledge artifacts. It reports observations, model outputs, and retrieved passages separately.


In [ ]:
from banking_observability.incident_evidence import assemble_investigation
packet = assemble_investigation(root / "data/raw", question="network timeout latency",
    as_of="2025-10-25 18:01:00", index_dir=root / "models/knowledge",
    log_model_dir=root / "models/log_clustering",
    anomaly_model_dir=root / "models/detection/monitoring_anomaly")
print(packet["monitoring_detector"]["review_required"])
print([(hit["document_id"], hit["score"]) for hit in packet["runbook_retrieval"]["results"]])
print(packet["interpretation"]["status"])

## Next layer

This notebook stops at evidence collection. The planned LangGraph workflow will need to cite the source records and leave unsupported causes unresolved.
